In [2]:
cd /content/drive/MyDrive/cv_22641171_NgoTruongDinh/lab_bo_sung_10 03 2026

/content/drive/MyDrive/cv_22641171_NgoTruongDinh/lab_bo_sung_10 03 2026


In [ ]:
cp -r '/content/drive/MyDrive/cv_22641171_NgoTruongDinh/lab_bo_sung_10 03 2026/new_Data/Covid19-Pneumonia-Normal Chest X-Ray Images Dataset.zip' /content

In [ ]:
!unzip -q '/content/Covid19-Pneumonia-Normal Chest X-Ray Images Dataset.zip' -d /content/dataset
!ls /content/dataset | head

COVID
NORMAL
PNEUMONIA


In [ ]:
import os

root = "/content/dataset"
valid_exts = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")

total = 0

for cls in os.listdir(root):
    cls_path = os.path.join(root, cls)
    if os.path.isdir(cls_path):
        n = sum(
            1 for f in os.listdir(cls_path)
            if f.lower().endswith(valid_exts)
        )
        print(f"{cls}: {n}")
        total += n

print("Tổng:", total)

NORMAL: 1802
PNEUMONIA: 1800
COVID: 1626
Tổng: 5228


In [ ]:
import os

import process_data

SRC_TRAIN = "/content/dataset"

OUT_ROOT = "/content/drive/MyDrive/cv_22641171_NgoTruongDinh/lab_bo_sung_10 03 2026/new_Data/new_dataset_processed"




train_classes = process_data.preprocess_split_mp(SRC_TRAIN,OUT_ROOT, max_workers=8)

print("Classes:", train_classes)

Preprocess dataset (workers=8):   0%|          | 0/5228 [00:00<?, ?it/s]

Done dataset: 5228/5228 files ok. Saved to: /content/drive/MyDrive/cv_22641171_NgoTruongDinh/lab_bo_sung_10 03 2026/new_Data/new_dataset_processed
Classes: ['COVID', 'NORMAL', 'PNEUMONIA']


In [ ]:
cp -r '/content/drive/MyDrive/cv_22641171_NgoTruongDinh/lab_bo_sung_10 03 2026/new_Data/new_dataset_processed' /content/data_processed

In [ ]:
import os

root = "/content/data_processed"
valid_exts = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")

total = 0

for cls in os.listdir(root):
    cls_path = os.path.join(root, cls)
    if os.path.isdir(cls_path):
        n = sum(
            1 for f in os.listdir(cls_path)
            if f.lower().endswith(valid_exts)
        )
        print(f"{cls}: {n}")
        total += n

print("Tổng:", total)

NORMAL: 1802
PNEUMONIA: 1800
COVID: 1626
Tổng: 5228


In [ ]:
import os
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mp

import numpy as np
from PIL import Image
from torchvision.datasets import ImageFolder
from tqdm.auto import tqdm

import encryption

SRC_ROOT = "/content/data_processed"
OUT_ROOT = "/content/new_dataset_encrypted"

IMG_SIZE = 224
SAVE_FMT = "PNG"
K = 3
BLOCK_XOR = 4
BLOCK_SHUFFLE = 16
SEED = 2024


def build_tasks(src_dir, out_dir):
    src_dir = str(src_dir)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    ds = ImageFolder(src_dir)
    classes = ds.classes

    tasks = []
    ext = ".jpg" if SAVE_FMT.upper() == "JPEG" else ".png"

    for path, label in ds.samples:
        cls = classes[label]
        stem = Path(path).stem
        dst = out_dir / cls / f"{stem}{ext}"
        tasks.append((path, str(dst)))

    return tasks, classes, ds.class_to_idx


def preload_images(tasks):
    """
    Load toàn bộ ảnh vào RAM trước để giảm bottleneck I/O.
    Trả về list[(img_array, dst_path)].
    """
    loaded = []

    for src_path, dst_path in tqdm(tasks, desc="Preloading to RAM"):
        dst_path = Path(dst_path)

        # nếu file đã tồn tại thì bỏ qua luôn
        if dst_path.exists():
            loaded.append((None, str(dst_path)))
            continue

        try:
            with Image.open(src_path) as im:
                im = im.convert("RGB")
                if im.size != (IMG_SIZE, IMG_SIZE):
                    im = im.resize((IMG_SIZE, IMG_SIZE), resample=Image.BILINEAR)
                img = np.array(im, dtype=np.uint8)  # HWC uint8

            loaded.append((img, str(dst_path)))
        except Exception:
            loaded.append((None, str(dst_path)))

    return loaded


def _encrypt_and_save(item):
    """
    Worker process:
    - nhận img đã nằm sẵn trong RAM
    - encrypt
    - save
    """
    img, dst_path = item
    try:
        if img is None:
            return False

        dst_path = Path(dst_path)
        dst_path.parent.mkdir(parents=True, exist_ok=True)

        enc = encryption.advanced_learnable_encrypt(
            img,
            k=K,
            block_xor=BLOCK_XOR,
            block_shuffle=BLOCK_SHUFFLE,
            seed=SEED
        )

        Image.fromarray(enc).save(dst_path, format=SAVE_FMT, optimize=False)
        return True
    except Exception:
        return False


def encrypt_split_mp_preload(src_dir, out_dir, max_workers=None, chunksize=64):
    tasks, classes, class_to_idx = build_tasks(src_dir, out_dir)

    # preload toàn bộ vào RAM
    loaded = preload_images(tasks)

    # chỉ giữ các ảnh chưa tồn tại + load OK
    loaded = [(img, dst) for img, dst in loaded if img is not None and not Path(dst).exists()]

    total = len(loaded)
    if total == 0:
        print(f"Done {Path(src_dir).name}: nothing to process -> {out_dir}")
        print("class_to_idx:", class_to_idx)
        return classes

    if max_workers is None:
        max_workers = os.cpu_count() or 4

    with ProcessPoolExecutor(max_workers=max_workers, mp_context=mp.get_context("fork")) as ex:
        results = list(
            tqdm(
                ex.map(_encrypt_and_save, loaded, chunksize=chunksize),
                total=total,
                desc=f"Encrypt {Path(src_dir).name} (workers={max_workers})"
            )
        )

    ok = sum(bool(x) for x in results)

    print(f"Done {Path(src_dir).name}: {ok}/{total} ok -> {out_dir}")
    print("class_to_idx:", class_to_idx)
    return classes


if __name__ == "__main__":
    workers = os.cpu_count() or 4

    train_classes = encrypt_split_mp_preload(
        SRC_ROOT,
        OUT_ROOT,
        max_workers=workers,
        chunksize=64
    )

    print("Classes:", train_classes)

Preloading to RAM:   0%|          | 0/5228 [00:00<?, ?it/s]

Encrypt data_processed (workers=12):   0%|          | 0/5228 [00:00<?, ?it/s]

Done data_processed: 5228/5228 ok -> /content/new_dataset_encrypted
class_to_idx: {'COVID': 0, 'NORMAL': 1, 'PNEUMONIA': 2}
Classes: ['COVID', 'NORMAL', 'PNEUMONIA']


In [ ]:
import os

root = "/content/drive/MyDrive/cv_22641171_NgoTruongDinh/lab_bo_sung_10 03 2026/new_Data/new_dataset_encrypted/new_dataset_encrypted"
valid_exts = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")

total = 0

for cls in os.listdir(root):
    cls_path = os.path.join(root, cls)
    if os.path.isdir(cls_path):
        n = sum(
            1 for f in os.listdir(cls_path)
            if f.lower().endswith(valid_exts)
        )
        print(f"{cls}: {n}")
        total += n

print("Tổng:", total)

COVID: 1626
NORMAL: 1802
PNEUMONIA: 1800
Tổng: 5228


In [ ]:
cp -r '/content/new_dataset_encrypted' '/content/drive/MyDrive/cv_22641171_NgoTruongDinh/lab_bo_sung_10 03 2026/new_Data/new_dataset_encrypted'

In [ ]:
import shutil

src_folder = "/content/new_dataset_processed"
zip_path = "/content/new_dataset_processed"

shutil.make_archive(zip_path, 'zip', src_folder)

print("Đã tạo:", zip_path + ".zip")

Đã tạo: /content/new_dataset_processed.zip


In [ ]:
import shutil

src_folder = "/content/new_dataset_processed"
zip_path = "/content/new_dataset_processed"

shutil.make_archive(zip_path, 'zip', src_folder)

print("Đã tạo:", zip_path + ".zip")

#TANAKA

In [2]:
!cp -r '/content/drive/MyDrive/cv_22641171_NgoTruongDinh/lab_bo_sung_10 03 2026/new_Data/new_dataset_processed' /content/data_processed

In [3]:
import os
import cv2
import numpy as np
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm

# =========================
# CONFIG
# =========================
KEY = 8888
BLOCK_SIZE = 4

INPUT_ROOT = "/content/data_processed"          # folder gốc đầu vào
OUTPUT_ROOT = "imgs_tanaka"  # folder gốc đầu ra
NUM_WORKERS = max(1, os.cpu_count() - 1)

VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

os.makedirs(OUTPUT_ROOT, exist_ok=True)

# =========================
# TANAKA-STYLE BLOCK SCRAMBLE
# =========================
def tanaka_encrypt_block_scramble(img, key=8888, block_size=4):
    """
    Mã hóa kiểu block scrambling với seed cố định.
    Hỗ trợ ảnh grayscale hoặc RGB.
    """
    if img is None:
        raise ValueError("Không đọc được ảnh đầu vào.")

    gray_input = False
    if img.ndim == 2:
        img = img[:, :, None]
        gray_input = True

    h, w, c = img.shape

    # Cắt phần dư để chia hết cho block_size
    new_h = (h // block_size) * block_size
    new_w = (w // block_size) * block_size

    if new_h == 0 or new_w == 0:
        raise ValueError(
            f"Ảnh quá nhỏ so với block_size={block_size}, shape={img.shape}"
        )

    img_crop = img[:new_h, :new_w].copy()

    nbh = new_h // block_size
    nbw = new_w // block_size
    num_blocks = nbh * nbw

    # Tách block
    blocks = []
    for by in range(nbh):
        for bx in range(nbw):
            y0 = by * block_size
            x0 = bx * block_size
            block = img_crop[y0:y0 + block_size, x0:x0 + block_size, :]
            blocks.append(block)

    # Sinh hoán vị theo key
    rng = np.random.default_rng(seed=key)
    perm = rng.permutation(num_blocks)

    # Ghép block theo thứ tự mới
    enc = np.zeros_like(img_crop)
    for out_idx, src_idx in enumerate(perm):
        out_by = out_idx // nbw
        out_bx = out_idx % nbw
        y0 = out_by * block_size
        x0 = out_bx * block_size
        enc[y0:y0 + block_size, x0:x0 + block_size, :] = blocks[src_idx]

    if gray_input:
        enc = enc[:, :, 0]

    return enc

# =========================
# COLLECT FILES
# =========================
def collect_image_files(input_root):
    input_root = Path(input_root)
    files = []

    for p in input_root.rglob("*"):
        if p.is_file() and p.suffix.lower() in VALID_EXTS:
            files.append(p)

    return sorted(files)

# =========================
# PROCESS ONE IMAGE
# =========================
def process_one(args):
    src_path, input_root, output_root, key, block_size = args
    src_path = Path(src_path)
    input_root = Path(input_root)
    output_root = Path(output_root)

    try:
        # relative path để giữ nguyên tree
        rel_path = src_path.relative_to(input_root)
        dst_path = output_root / rel_path
        dst_path.parent.mkdir(parents=True, exist_ok=True)

        # đọc ảnh
        img_bgr = cv2.imread(str(src_path), cv2.IMREAD_COLOR)
        if img_bgr is None:
            return (False, str(src_path), "Không đọc được ảnh")

        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        # mã hóa
        enc_rgb = tanaka_encrypt_block_scramble(
            img_rgb,
            key=key,
            block_size=block_size
        )

        # lưu
        enc_bgr = cv2.cvtColor(enc_rgb, cv2.COLOR_RGB2BGR)
        ok = cv2.imwrite(str(dst_path), enc_bgr)

        if not ok:
            return (False, str(src_path), "cv2.imwrite thất bại")

        return (True, str(src_path), str(dst_path))

    except Exception as e:
        return (False, str(src_path), str(e))

# =========================
# MAIN
# =========================
def main():
    files = collect_image_files(INPUT_ROOT)

    if len(files) == 0:
        raise FileNotFoundError(f"Không tìm thấy ảnh nào trong: {INPUT_ROOT}")

    print(f"Tìm thấy {len(files)} ảnh")
    print(f"Input root : {INPUT_ROOT}")
    print(f"Output root: {OUTPUT_ROOT}")
    print(f"Workers    : {NUM_WORKERS}")

    tasks = [
        (str(fp), str(INPUT_ROOT), str(OUTPUT_ROOT), KEY, BLOCK_SIZE)
        for fp in files
    ]

    success_count = 0
    fail_count = 0
    failed_items = []

    with ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
        futures = [executor.submit(process_one, task) for task in tasks]

        for fut in tqdm(as_completed(futures), total=len(futures), desc="Encrypting"):
            ok, src, msg = fut.result()
            if ok:
                success_count += 1
            else:
                fail_count += 1
                failed_items.append((src, msg))

    print("\n===== DONE =====")
    print(f"Thành công: {success_count}")
    print(f"Thất bại  : {fail_count}")

    if failed_items:
        print("\nCác file lỗi:")
        for src, err in failed_items[:20]:
            print(f"- {src} -> {err}")
        if len(failed_items) > 20:
            print(f"... và {len(failed_items) - 20} file lỗi khác")

if __name__ == "__main__":
    main()

Tìm thấy 5228 ảnh
Input root : /content/data_processed
Output root: imgs_tanaka
Workers    : 11


Encrypting:   0%|          | 0/5228 [00:00<?, ?it/s]


===== DONE =====
Thành công: 5228
Thất bại  : 0


# SKK

In [4]:
import os
import cv2
import numpy as np
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm

# =========================
# CONFIG
# =========================
INPUT_ROOT = "/content/data_processed"               # folder ảnh gốc
OUTPUT_ROOT = "imgs_skk"    # folder ảnh mã hóa cuối cùng để train

SEED = 8888
BLOCK_SIZE = 4
NUM_WORKERS = max(1, os.cpu_count() - 1)

VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

os.makedirs(OUTPUT_ROOT, exist_ok=True)

# =========================
# COLLECT FILES
# =========================
def collect_image_files(input_root):
    input_root = Path(input_root)
    files = []
    for p in input_root.rglob("*"):
        if p.is_file() and p.suffix.lower() in VALID_EXTS:
            files.append(p)
    return sorted(files)

# =========================
# STEP 1: NEGATIVE-POSITIVE
# =========================
def negative_positive_transform(img_rgb, rng):
    h, w, c = img_rgb.shape
    negpos_mask = rng.random((h, w, 3)) >= 0.5

    out = img_rgb.copy()
    out[negpos_mask] = np.bitwise_xor(out[negpos_mask], 255).astype(np.uint8)
    return out

# =========================
# STEP 2: CHANNEL SHUFFLE
# =========================
def channel_shuffle(img_rgb, rng):
    perms = np.array([
        [0, 1, 2],  # RGB
        [0, 2, 1],  # RBG
        [1, 0, 2],  # GRB
        [1, 2, 0],  # GBR
        [2, 0, 1],  # BRG
        [2, 1, 0],  # BGR
    ], dtype=np.int64)

    h, w, _ = img_rgb.shape
    shuffle_idx = rng.integers(0, 6, size=(h, w))
    selected_perms = perms[shuffle_idx]   # (h, w, 3)

    out = np.take_along_axis(img_rgb, selected_perms, axis=2)
    return out

# =========================
# STEP 3: BLOCK SCRAMBLE
# =========================
def block_scramble(img_rgb, rng, block_size=4):
    if img_rgb.ndim == 2:
        img_rgb = img_rgb[:, :, None]
        gray_input = True
    else:
        gray_input = False

    h, w, c = img_rgb.shape

    new_h = (h // block_size) * block_size
    new_w = (w // block_size) * block_size

    if new_h == 0 or new_w == 0:
        raise ValueError(
            f"Ảnh quá nhỏ cho block_size={block_size}, shape={img_rgb.shape}"
        )

    img_crop = img_rgb[:new_h, :new_w].copy()

    nbh = new_h // block_size
    nbw = new_w // block_size
    num_blocks = nbh * nbw

    blocks = []
    for by in range(nbh):
        for bx in range(nbw):
            y0 = by * block_size
            x0 = bx * block_size
            block = img_crop[y0:y0 + block_size, x0:x0 + block_size, :]
            blocks.append(block)

    perm = rng.permutation(num_blocks)

    enc = np.zeros_like(img_crop)
    for out_idx, src_idx in enumerate(perm):
        out_by = out_idx // nbw
        out_bx = out_idx % nbw
        y0 = out_by * block_size
        x0 = out_bx * block_size
        enc[y0:y0 + block_size, x0:x0 + block_size, :] = blocks[src_idx]

    if gray_input:
        enc = enc[:, :, 0]

    return enc

# =========================
# FULL SKK ENCRYPTION
# =========================
def skk_encrypt_final(img_rgb, seed=8888, block_size=4):
    rng = np.random.default_rng(seed)

    x = negative_positive_transform(img_rgb, rng)
    x = channel_shuffle(x, rng)
    x = block_scramble(x, rng, block_size=block_size)

    return x

# =========================
# PROCESS ONE IMAGE
# =========================
def process_one(args):
    src_path, input_root, output_root, seed, block_size = args
    src_path = Path(src_path)
    input_root = Path(input_root)
    output_root = Path(output_root)

    try:
        rel_path = src_path.relative_to(input_root)
        dst_path = output_root / rel_path
        dst_path.parent.mkdir(parents=True, exist_ok=True)

        img_bgr = cv2.imread(str(src_path), cv2.IMREAD_COLOR)
        if img_bgr is None:
            return (False, str(src_path), "Không đọc được ảnh")

        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        enc_rgb = skk_encrypt_final(
            img_rgb,
            seed=seed,
            block_size=block_size
        )

        ok = cv2.imwrite(str(dst_path), cv2.cvtColor(enc_rgb, cv2.COLOR_RGB2BGR))
        if not ok:
            return (False, str(src_path), f"Lưu thất bại: {dst_path}")

        return (True, str(src_path), str(dst_path))

    except Exception as e:
        return (False, str(src_path), str(e))

# =========================
# MAIN
# =========================
def main():
    files = collect_image_files(INPUT_ROOT)

    if len(files) == 0:
        raise FileNotFoundError(f"Không tìm thấy ảnh nào trong: {INPUT_ROOT}")

    print(f"Tìm thấy {len(files)} ảnh")
    print(f"Input root : {INPUT_ROOT}")
    print(f"Output root: {OUTPUT_ROOT}")
    print(f"Workers    : {NUM_WORKERS}")
    print(f"Seed       : {SEED}")
    print(f"Block size : {BLOCK_SIZE}")

    tasks = [
        (str(fp), str(INPUT_ROOT), str(OUTPUT_ROOT), SEED, BLOCK_SIZE)
        for fp in files
    ]

    success_count = 0
    fail_count = 0
    failed_items = []

    with ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
        futures = [executor.submit(process_one, task) for task in tasks]

        for fut in tqdm(as_completed(futures), total=len(futures), desc="Encrypting"):
            ok, src, msg = fut.result()
            if ok:
                success_count += 1
            else:
                fail_count += 1
                failed_items.append((src, msg))

    print("\n===== DONE =====")
    print(f"Thành công: {success_count}")
    print(f"Thất bại  : {fail_count}")

    if failed_items:
        print("\nCác file lỗi:")
        for src, err in failed_items[:20]:
            print(f"- {src} -> {err}")
        if len(failed_items) > 20:
            print(f"... và {len(failed_items) - 20} file lỗi khác")

if __name__ == "__main__":
    main()

Tìm thấy 5228 ảnh
Input root : /content/data_processed
Output root: imgs_skk
Workers    : 11
Seed       : 8888
Block size : 4


Encrypting:   0%|          | 0/5228 [00:00<?, ?it/s]


===== DONE =====
Thành công: 5228
Thất bại  : 0


# Huang

In [1]:
!cp -r '/content/drive/MyDrive/cv_22641171_NgoTruongDinh/lab_bo_sung_10 03 2026/new_Data/new_dataset_processed' /content/data_processed

In [4]:
import os
import cv2
import zlib
import numpy as np
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm

# =========================================================
# Huang et al. 2022:
# Privacy-Preserving Deep Learning With Learnable Image Encryption on Medical Images
# Proposed scheme = SKK-like encryption + block statistical smoothing
# Multi-process + recursive folder support
# =========================================================

IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}

# =========================
# CONFIG
# =========================
INPUT_PATH = "/content/data_processed"                # file hoặc thư mục gốc
OUTPUT_PATH = "imgs_huang2022"     # thư mục output
BLOCK_SIZE = 4                     # MRI thường 4, X-ray có thể 6
BITS = 8
BASE_SEED = 42                     # để None nếu muốn random hoàn toàn
NUM_WORKERS = max(1, os.cpu_count() - 1)

# 6 channel permutations
PERM_TABLE = np.array([
    [0, 1, 2],  # RGB
    [0, 2, 1],  # RBG
    [1, 0, 2],  # GRB
    [1, 2, 0],  # GBR
    [2, 0, 1],  # BRG
    [2, 1, 0],  # BGR
], dtype=np.int64)


def _to_uint8(img: np.ndarray) -> np.ndarray:
    if img.dtype == np.uint8:
        return img

    arr = img.astype(np.float32)
    mn, mx = arr.min(), arr.max()

    if mx <= mn:
        return np.zeros_like(arr, dtype=np.uint8)

    arr = (arr - mn) / (mx - mn) * 255.0
    return np.rint(arr).clip(0, 255).astype(np.uint8)


def _read_as_rgb(input_file: Path) -> np.ndarray:
    img = cv2.imread(str(input_file), cv2.IMREAD_UNCHANGED)
    if img is None:
        raise FileNotFoundError(f"Không đọc được ảnh: {input_file}")

    img = _to_uint8(img)

    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    elif img.ndim == 3 and img.shape[2] == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    elif img.ndim == 3 and img.shape[2] == 4:
        img = cv2.cvtColor(img, cv2.COLOR_BGRA2RGB)
    else:
        raise ValueError(f"Định dạng ảnh không hỗ trợ: shape={img.shape}, dtype={img.dtype}")

    return img


def _save_rgb(output_file: Path, img_rgb: np.ndarray) -> None:
    output_file.parent.mkdir(parents=True, exist_ok=True)
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    ok = cv2.imwrite(str(output_file), img_bgr)
    if not ok:
        raise IOError(f"Không ghi được ảnh ra: {output_file}")


def _negative_positive_transform(img_rgb: np.ndarray, bits: int, rng: np.random.Generator) -> np.ndarray:
    if bits != 8:
        raise ValueError("Code này thiết kế cho ảnh 8-bit, nên bits nên là 8.")

    h, w, c = img_rgb.shape
    xor_value = (1 << bits) - 1  # 255

    knp = rng.integers(0, 2, size=(h, w, c), dtype=np.uint8)
    out = np.bitwise_xor(img_rgb, knp * xor_value)
    return out.astype(np.uint8)


def _color_shuffle(img_rgb: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    h, w, _ = img_rgb.shape
    kcs = rng.integers(0, 6, size=(h, w))
    perms = PERM_TABLE[kcs]
    out = np.take_along_axis(img_rgb, perms, axis=2)
    return out.astype(np.uint8)


def _block_statistical_smoothing(img_rgb: np.ndarray, block_size: int, rng: np.random.Generator) -> np.ndarray:
    if block_size < 1:
        raise ValueError("block_size phải >= 1")

    h, w, _ = img_rgb.shape
    out = img_rgb.copy()

    for y in range(0, h, block_size):
        for x in range(0, w, block_size):
            block = out[y:y + block_size, x:x + block_size]

            kss = int(rng.integers(0, 4))  # 0 mean, 1 median, 2 max, 3 min

            if kss == 0:
                value = np.rint(block.mean(axis=(0, 1))).clip(0, 255).astype(np.uint8)
            elif kss == 1:
                value = np.rint(np.median(block, axis=(0, 1))).clip(0, 255).astype(np.uint8)
            elif kss == 2:
                value = block.max(axis=(0, 1)).astype(np.uint8)
            else:
                value = block.min(axis=(0, 1)).astype(np.uint8)

            block[:] = value.reshape(1, 1, 3)

    return out


def encrypt_huang2022_array(
    img_rgb: np.ndarray,
    block_size: int = 4,
    bits: int = 8,
    seed: int | None = 42
) -> np.ndarray:
    rng = np.random.default_rng(seed)

    img_rgb = _to_uint8(img_rgb)
    if img_rgb.ndim != 3 or img_rgb.shape[2] != 3:
        raise ValueError("img_rgb phải có shape (H, W, 3)")

    out = _negative_positive_transform(img_rgb, bits=bits, rng=rng)
    out = _color_shuffle(out, rng=rng)
    out = _block_statistical_smoothing(out, block_size=block_size, rng=rng)

    return out.astype(np.uint8)


def _make_per_image_seed(rel_path: Path, base_seed: int | None):
    if base_seed is None:
        return None
    rel_str = rel_path.as_posix()
    crc = zlib.crc32(rel_str.encode("utf-8")) & 0xFFFFFFFF
    return (int(base_seed) + crc) % (2**32)


def _encrypt_one_file(
    input_file: Path,
    output_file: Path,
    block_size: int,
    bits: int,
    seed: int | None
) -> None:
    img_rgb = _read_as_rgb(input_file)
    enc_rgb = encrypt_huang2022_array(
        img_rgb,
        block_size=block_size,
        bits=bits,
        seed=seed
    )
    _save_rgb(output_file, enc_rgb)


def _collect_image_files(input_path: Path):
    if input_path.is_file():
        if input_path.suffix.lower() not in IMG_EXTS:
            raise ValueError(f"File không phải ảnh hỗ trợ: {input_path}")
        return [input_path]

    files = sorted([
        p for p in input_path.rglob("*")
        if p.is_file() and p.suffix.lower() in IMG_EXTS
    ])

    if len(files) == 0:
        raise ValueError(f"Không tìm thấy ảnh nào trong thư mục: {input_path}")

    return files


def _process_one(args):
    src, input_root, output_root, block_size, bits, base_seed = args
    src = Path(src)
    input_root = Path(input_root)
    output_root = Path(output_root)

    try:
        if input_root.is_file():
            rel = Path(src.name)
        else:
            rel = src.relative_to(input_root)

        dst = output_root / rel
        dst.parent.mkdir(parents=True, exist_ok=True)

        per_image_seed = _make_per_image_seed(rel, base_seed)

        _encrypt_one_file(
            input_file=src,
            output_file=dst,
            block_size=block_size,
            bits=bits,
            seed=per_image_seed
        )

        return True, str(src), str(dst)

    except Exception as e:
        return False, str(src), str(e)


def encrypt_huang2022(
    input_path: str,
    output_path: str,
    block_size: int = 4,
    bits: int = 8,
    seed: int | None = 42,
    num_workers: int = 1
) -> None:
    input_path = Path(input_path)
    output_path = Path(output_path)

    if not input_path.exists():
        raise FileNotFoundError(f"Không tồn tại input_path: {input_path}")

    files = _collect_image_files(input_path)
    output_path.mkdir(parents=True, exist_ok=True)

    tasks = [
        (str(src), str(input_path), str(output_path), block_size, bits, seed)
        for src in files
    ]

    print(f"Tìm thấy {len(files)} ảnh")
    print(f"Input       : {input_path}")
    print(f"Output      : {output_path}")
    print(f"Block size  : {block_size}")
    print(f"Bits        : {bits}")
    print(f"Base seed   : {seed}")
    print(f"Workers     : {num_workers}")

    success_count = 0
    fail_count = 0
    failed_items = []

    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        futures = [executor.submit(_process_one, task) for task in tasks]

        for fut in tqdm(as_completed(futures), total=len(futures), desc="Encrypting"):
            ok, src, msg = fut.result()
            if ok:
                success_count += 1
            else:
                fail_count += 1
                failed_items.append((src, msg))

    print("\n===== DONE =====")
    print(f"Thành công: {success_count}")
    print(f"Thất bại  : {fail_count}")

    if failed_items:
        print("\nCác file lỗi:")
        for src, err in failed_items[:20]:
            print(f"- {src} -> {err}")
        if len(failed_items) > 20:
            print(f"... và {len(failed_items) - 20} file lỗi khác")


if __name__ == "__main__":
    encrypt_huang2022(
        input_path=INPUT_PATH,
        output_path=OUTPUT_PATH,
        block_size=BLOCK_SIZE,
        bits=BITS,
        seed=BASE_SEED,
        num_workers=NUM_WORKERS
    )

Tìm thấy 5228 ảnh
Input       : /content/data_processed
Output      : imgs_huang2022
Block size  : 4
Bits        : 8
Base seed   : 42
Workers     : 11


Encrypting:   0%|          | 0/5228 [00:00<?, ?it/s]


===== DONE =====
Thành công: 5228
Thất bại  : 0
